# 多模态全栈评测、企业级部署与商业化交付

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节定位 · 收官课：从"能跑"到"可交付" | — |
| **Part 2** | 评测体系结构与模块地图 | `eval/**` |
| **Part 3** | 四套件与注册表 | `eval/suites/__init__.py` |
| **Part 4** | 指标库：CER / 检索 / 字段 / 描述 / 统计 | `eval/metrics.py` |
| **Part 5** | 公共流水线：用例 / 限流 / 阈值 / 装配 | `eval/suites/_common.py` |
| **Part 6** | 四个套件走读 | `eval/suites/{asr,tts,vlm,retrieval}.py` |
| **Part 7** | 数据与素材：jsonl 用例 + 合成图 | `eval/datasets/*.jsonl`、`eval/images.py` |
| **Part 8** | 编排与持久化 | `eval/runner.py`、`eval/store.py` |
| **Part 9** | 管理端接口 + 回归测试（简） | `core/api/admin_api.py`、`tests/*` |
| **Part 10** | 容器化部署与交付 | `deploy/*`、`requirements.txt`、`.dockerignore` |
| **Part 11** | 实战任务 + 随堂练习（含答案）+ 收官小结 | 多模块联合 |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解和代码注释，
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。
>
> 这是六节课的**收官课**。前面把系统做出来、跑通了，本节回答交付前的最后一问：
> **它到底好不好用（评测）？能不能一键部署（容器化）？能不能交给别人（文档与交付）？**

---
## Part 1：本节在架构中的位置 —— 从"能跑"到"可交付"
---

### 1.1 收官课的三条主线

| 主线 | 要回答的问题 | 关键代码 |
|------|--------------|----------|
| **评测** | 效果到底如何？有没有量化、可复现的证据？ | `eval/**` |
| **回归测试** | 改了代码，怎么保证没把老功能改坏？ | `tests/**` |
| **部署与交付** | 换台机器怎么一键起来？数据会不会丢？ | `deploy/**` |

### 1.2 为什么"跑通一次"不等于"可交付"

一个能演示的系统，离可交付还差三样东西：

1. **可度量**：说"效果好"没有用，要能说出 ASR 字错率、检索 Recall@3 是多少，
   以及和基线比是变好还是变坏；
2. **可回归**：修复一个 Bug 时，不能让另一个功能悄悄坏掉——需要自动化测试兜底；
3. **可复现**：别人拿到代码能一条命令起起来，重启后数据还在。

### 1.3 本节在工程结构中的位置

| 层 | 目录 | 本节涉及的代码 |
|---|---|---|
| 路由层 | `core/api/` | `admin_api.py`：`/api/admin/eval/*`（发起评测 / 查看 / 设基线 / 删除） |
| 基础设施层 | `core/infra/` | `scheduler.py`（超时关闭）、`db.py`（业务库持久化） |
| 智能编排层 | `core/ai/` | `llm.extract_json()`：被评测的 LLM-as-judge 复用 |
| 多模态 RAG 层 | `core/rag/` | 被评测的能力：`multimodal.py`（ASR/TTS/VLM）、`multimodal_rag.py`（检索） |
| 业务编排层 | `core/service/` | 被评测的业务：转人工、会话流转 |
| **评测层** | **`eval/`** | **四套件 + 指标 + 评判 + 落库 + 执行器** |
| **交付层** | **`deploy/`、`tests/`、`requirements.txt`** | **镜像 / 编排 / 回归测试 / 依赖锁定** |

### 1.4 交付验收标准（本项目的"封版线"）

```
① pytest 全绿                 —— 关键路径回归通过
② 四个评测套件能出报告         —— PASS/FAIL/SKIP 判定明确、可设基线对比
③ docker compose up -d 起全栈  —— :8000 / :3000 均可访问
④ 重启容器后数据不丢           —— ChromaDB / 媒体 / 业务库 / 工单库都在 data 卷内
```

### 1.5 三条贯穿全节的统一约定

1. **先定义"怎么算对"，再谈"效果好不好"**：评测的价值在于**可复现、可对比**，
   而不是刷一个好看的分数。
2. **"没测到"与"测了不达标"必须区分**：前者是 `SKIP`（环境/依赖问题），
   后者是 `FAIL`（质量问题）。混在一起会把环境问题误判成质量不达标。
3. **持久化数据必须落在卷内**：容器可以随便重建，`data/` 不能丢。

---
## Part 2：评测体系结构与模块地图
---

### 2.1 环境检查

In [1]:
# ============================================================
# Cell 1：环境检查（评测包可导入 + 关键依赖）
# ============================================================
import sys
import importlib

print("=" * 60)
print("  环境检查")
print("=" * 60)
print(f"  Python 版本: {sys.version.split()[0]}")
print(f"  解释器路径: {sys.executable}")
print()

for pkg, imp in {
    "pytest": "pytest",       # 回归测试
    "PIL": "PIL",             # 评测合成图
    "chromadb": "chromadb",   # 检索套件前置
    "pydantic": "pydantic",
}.items():
    try:
        importlib.import_module(imp)
        print(f"     [OK] {pkg}")
    except ImportError:
        print(f"     [NO] {pkg} 未安装")

print()
# 评测包本身可导入（说明 core.config 能正常加载）
from eval import metrics
from eval.suites import SUITES
print(f"  eval 包可导入；已注册套件：{list(SUITES)}")
print(f"  core.config 加载正常（.env 生效）")

  环境检查
  Python 版本: 3.13.7
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

     [OK] pytest
     [OK] PIL
     [OK] chromadb
     [OK] pydantic

  eval 包可导入；已注册套件：['asr', 'tts', 'vlm', 'retrieval']
  core.config 加载正常（.env 生效）


> 提示：`core/config.py` 用**仓库根绝对路径**加载 `.env`，因此无论 Jupyter 内核
> 的工作目录在哪，读到的都是本项目的 `.env`，不会因 CWD 变化而读错配置。

### 2.2 评测目录结构

`eval/` 一共 8 个文件 + 4 个数据集，分工非常清楚。

In [2]:
# ============================================================
# Cell 2：扫描 eval/ 目录结构与规模
# ============================================================
from pathlib import Path

print("=" * 64)
print("  eval/ 目录结构")
print("=" * 64)
for p in sorted(Path("eval").rglob("*")):
    if p.is_file() and "__pycache__" not in str(p) and p.suffix in (".py", ".jsonl"):
        lines = len(p.read_text(encoding="utf-8").splitlines())
        print(f"    {str(p.relative_to('.')):<42} {lines:>4} 行")
print()
print("  读法：")
print("    - metrics.py / suites/_common.py 是'纯函数与公共工具'，可离线单测；")
print("    - suites/{asr,tts,vlm,retrieval}.py 是四个可执行套件；")
print("    - runner.py 编排一次运行，store.py 负责落盘；")
print("    - judges.py 做 LLM-as-judge，images.py 生成合成评测图。")

  eval/ 目录结构
    eval\__init__.py                              5 行
    eval\datasets\caption_cases.jsonl             3 行
    eval\datasets\retrieval_cases.jsonl          10 行
    eval\datasets\speech_cases.jsonl             10 行
    eval\datasets\vlm_cases.jsonl                 3 行
    eval\images.py                              160 行
    eval\judges.py                              131 行
    eval\metrics.py                             305 行
    eval\runner.py                              131 行
    eval\store.py                               239 行
    eval\suites\__init__.py                      43 行
    eval\suites\_common.py                      194 行
    eval\suites\asr.py                          133 行
    eval\suites\retrieval.py                    162 行
    eval\suites\tts.py                          188 行
    eval\suites\vlm.py                          201 行

  读法：
    - metrics.py / suites/_common.py 是'纯函数与公共工具'，可离线单测；
    - suites/{asr,tts,vlm,retrieval}.py 是四个可执行套件；
    - runn

### 2.3 模块"说明书"：先读 docstring

每个评测模块开头都写清了自己的**指标口径与已知局限**。先读说明书，再看代码。

In [3]:
# ============================================================
# Cell 3：模块 docstring + 模块地图（先有说明书，再看细节）
# ============================================================
import ast
import eval.metrics as metrics_mod
import eval.store as store_mod
import eval.runner as runner_mod
import eval.judges as judges_mod
import eval.images as images_mod
import eval.suites._common as common_mod

for label, mod in [
    ("eval/metrics.py", metrics_mod),
    ("eval/suites/_common.py", common_mod),
    ("eval/runner.py", runner_mod),
    ("eval/store.py", store_mod),
    ("eval/judges.py", judges_mod),
    ("eval/images.py", images_mod),
]:
    print("=" * 70)
    print(f"  {label}")
    print("=" * 70)
    print((mod.__doc__ or "").strip())
    print()

  eval/metrics.py
评测指标库（纯函数，无网络 / 无副作用，便于离线单测）。

四类指标：
  - 文本错误率：CER / WER（ASR 识别准确率、TTS 反向可懂度）
  - 检索排序：Recall@k / Precision@k / Hit@k / MRR / nDCG@k
  - 结构化字段：归一化精确匹配 / 字段准确率（VLM 截图诊断）
  - 描述质量：关键要素命中率 / 余弦相似度（VLM 配图描述）

单测见 ``tests/test_eval_metrics.py``（不依赖任何外部服务）。

  eval/suites/_common.py
评测套件公共工具：用例加载、并发限流、阈值判定、结果装配。

四个套件（asr / tts / vlm / retrieval）都走同一条流水线：
「读用例 → 逐条执行 → 聚合指标 → 判定 PASS/FAIL/SKIP → 装配结果」。
本模块把这些共性抽出来，避免各套件重复造轮子。

关键概念：
  - **用例（case）**：一行 JSON，来自 ``eval/datasets/*.jsonl``。jsonl 即
    「每行一个独立 JSON 对象」，好处是能逐行流式解析、追加不破坏全文，
    且某一行坏了只丢这一行、不影响其余用例。
  - **skip 与 fail 的区别**：SKIP 表示「没法测」（依赖缺失、指标全为空），
    不算误报；FAIL 表示「确实测了但不达标」。两者必须区分，否则把环境问题
    当成质量问题，会误伤被测能力。

  eval/runner.py
评测编排（由 admin 接口在后台任务中调用，**不提供命令行入口**）。

生命周期：
    create_run(suites)  建记录（status=running）→ 返回 run_id，接口立即回给前端；
    execute(run_id)     后台逐套件执行，边跑边落盘（前端轮询进度）；
    finalize            收尾（status=done / failed）并写入汇总。

  eval/store.py
评测运行的持久化（文件式，零额外依赖）。

    data/eval/runs/{run_id}.json   单次评测的完

In [4]:
# ============================================================
# Cell 4：eval/ 各文件的顶层类 / 函数地图
# ============================================================
FILES = [
    "eval/metrics.py",
    "eval/suites/_common.py",
    "eval/suites/asr.py",
    "eval/suites/tts.py",
    "eval/suites/vlm.py",
    "eval/suites/retrieval.py",
    "eval/runner.py",
    "eval/store.py",
    "eval/judges.py",
    "eval/images.py",
]

for path in FILES:
    tree = ast.parse(Path(path).read_text(encoding="utf-8"))
    items = []
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            items.append((node.lineno, "class", node.name))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            kind = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
            items.append((node.lineno, kind, node.name))
    print("=" * 70)
    print(f"  {path}   （{len(items)} 个顶层类/函数）")
    print("=" * 70)
    for line, kind, name in sorted(items):
        mark = "  (内部)" if name.startswith("_") else ""
        print(f"  [{kind:^9}] {name:<30} (第 {line} 行){mark}")
    print()

  eval/metrics.py   （22 个顶层类/函数）
  [   def   ] _cjk_run_to_arabic             (第 49 行)  (内部)
  [   def   ] normalize_text                 (第 71 行)
  [   def   ] _tokenize_en                   (第 81 行)  (内部)
  [   def   ] edit_distance                  (第 91 行)
  [   def   ] char_error_rate                (第 115 行)
  [   def   ] word_error_rate                (第 127 行)
  [   def   ] corpus_cer                     (第 136 行)
  [   def   ] sentence_accuracy              (第 150 行)
  [   def   ] recall_at_k                    (第 164 行)
  [   def   ] precision_at_k                 (第 172 行)
  [   def   ] hit_at_k                       (第 179 行)
  [   def   ] reciprocal_rank                (第 184 行)
  [   def   ] mrr                            (第 193 行)
  [   def   ] ndcg_at_k                      (第 202 行)
  [   def   ] exact_match                    (第 216 行)
  [   def   ] field_accuracy                 (第 223 行)
  [   def   ] substring_match                (第 232 行)
  [   def   ] key_point_

---
## Part 3：四套件与注册表 —— 插件式设计
---

### 3.1 为什么要"套件化"

多模态有四种完全不同的能力（听 / 说 / 看 / 查）。如果把它们写成一个巨大的评测函数，
改一个指标就要动全身。本项目的做法是：**每个套件是一个独立模块**，
对外只暴露三个约定接口：

```python
SUITE = "asr"          # 套件标识（API 里用它指名）
TITLE = "语音识别（ASR）字错率"
async def run(limit=None) -> dict:   # 执行并返回统一结构
    ...
```

`runner` 只认这三个约定，**不关心套件内部怎么实现**——这就是"插件目录"的思路。
新增一种能力评测，只需写一个同构模块并登记进 `SUITES`。

In [5]:
# ============================================================
# Cell 5：套件注册表源码 + normalize 实测
# ============================================================
import inspect
import eval.suites as suites_pkg

print(inspect.getsource(suites_pkg))

print("实测 normalize（校验 + 按默认顺序排序）：")
print("  normalize(None)                ->", suites_pkg.normalize(None))
print("  normalize(['retrieval','asr']) ->", suites_pkg.normalize(["retrieval", "asr"]))
print("  （注意：传入顺序被打乱，输出恒按 DEFAULT_ORDER 排列，保证报告顺序稳定）")
try:
    suites_pkg.normalize(["nope"])
except ValueError as exc:
    print("  normalize(['nope'])            -> 抛 ValueError：", exc)

"""评测套件注册表：套件名 → 模块（模块需暴露 SUITE / TITLE / async run()）。

这里相当于「插件目录」：runner 只认这三个约定接口，不关心具体套件内部实现，
新增套件时只需写一个同构模块并登记到 SUITES 即可。
"""
from __future__ import annotations

from eval.suites import asr, retrieval, tts, vlm

# 套件名 → 模块。键即 API 里使用的套件标识
SUITES = {
    asr.SUITE: asr,
    tts.SUITE: tts,
    vlm.SUITE: vlm,
    retrieval.SUITE: retrieval,
}

# 默认执行顺序：语音输入 → 语音输出 → 图文理解 → 检索
DEFAULT_ORDER: tuple[str, ...] = (asr.SUITE, tts.SUITE, vlm.SUITE, retrieval.SUITE)

# 套件名 → 展示标题（从各模块的 TITLE 汇总，供前端列表显示）
TITLES = {name: module.TITLE for name, module in SUITES.items()}


def normalize(names: list[str] | None) -> list[str]:
    """校验并归一化套件名列表；为空时返回默认全集。

    参数：
        names：调用方传入的套件名列表；None 或空表示「全部」。
    返回：
        按 DEFAULT_ORDER 排好序的子集列表。空传入返回默认全集。
    异常：
        ValueError：出现未知套件名时报错，避免静默跑空。
    """
    if not names:
        return list(DEFAULT_ORDER)
    # 按默认顺序取交集：既过滤掉非法名，又保证执行顺序稳定
    ordered = [n for n in DEFAULT_ORDER if n in set(names)]
    unknown = [n for n in names if n not 

---
## Part 4：指标库 `metrics.py` —— 纯函数，可离线测试
---

### 4.1 设计原则：指标是纯函数

`metrics.py` 里**没有网络、没有文件 IO、没有副作用**——输入两个文本/两个列表，
输出一个数。好处是：**能被单元测试逐个钉死**（`tests/test_eval_metrics.py`），
也能在任何环境离线复现。

四类指标：

| 类别 | 函数 | 用在哪 |
|------|------|--------|
| 文本错误率 | `char_error_rate` / `word_error_rate` / `corpus_cer` | ASR 识别、TTS 反向可懂度 |
| 检索排序 | `recall_at_k` / `hit_at_k` / `reciprocal_rank` / `ndcg_at_k` | 检索套件 |
| 结构化字段 | `exact_match` / `substring_match` / `field_accuracy` | VLM 截图诊断 |
| 描述质量 | `key_point_hit_rate` / `cosine` | VLM 配图描述 |
| 统计工具 | `mean` / `percentile` / `pearson` | 时延、语速一致性 |

### 4.2 文本归一化：评测的第一道关口

**比对前必须先归一化**，否则会把"写法差异"误算成"识别错误"。
本项目要处理三类中文场景的坑：

| 场景 | 不归一化的后果 | 处理 |
|------|----------------|------|
| 全角/大小写 | `ＶＰＮ` vs `vpn` 判为不同 | NFKC 折全角 + 小写 |
| **中文数字** | ASR 把「619」转写成「六幺九」，CER 虚高 | 中文数字串整体转阿拉伯数字 |
| **小数点写法** | 口语「三点二」vs 书面「3.2」 | 夹在数字间的 `.`/`·`/`点` 统一剔除 |

第一条很关键：不做的话，"识别正确"会被判成"全错"，评测直接失真。

In [6]:
# ============================================================
# Cell 6：归一化实测（中文数字 / 全角 / 小数点）
# ============================================================
print("=" * 66)
print("  normalize_text 实测")
print("=" * 66)
samples = [
    "Hello, World!",
    "ＶＰＮ",
    "六幺九",
    "六百一十九",
    "十九",
    "一百二十",
    "三点二",
    "3.2",
    "零点五",
    "0.5",
    "重点",           # 「点」不在数字之间，不能被误伤
    "错误 619。",
    "端口 993",
    "0x80070005",
]
for s in samples:
    print(f"    {s!r:<18} -> {metrics.normalize_text(s)!r}")
print()
print("  结论：写法不同的同义文本，归一化后完全一致——这才是可比的评测。")

  normalize_text 实测
    'Hello, World!'    -> 'helloworld'
    'ＶＰＮ'              -> 'vpn'
    '六幺九'              -> '619'
    '六百一十九'            -> '619'
    '十九'               -> '19'
    '一百二十'             -> '120'
    '三点二'              -> '32'
    '3.2'              -> '32'
    '零点五'              -> '05'
    '0.5'              -> '05'
    '重点'               -> '重点'
    '错误 619。'          -> '错误619'
    '端口 993'           -> '端口993'
    '0x80070005'       -> '0x80070005'

  结论：写法不同的同义文本，归一化后完全一致——这才是可比的评测。


### 4.3 错率：编辑距离是核心

`edit_distance`（Levenshtein）算的是"把假设文本改成参考文本，最少要几次
替换/删除/插入"。除以参考长度就是错率：

```
CER = (S + D + I) / N      # 字级，N = 参考字数
WER = (S + D + I) / N      # 词级（英文按词切分）
```

- `char_error_rate`：单句 CER；`corpus_cer`：**语料级** CER（Σ编辑距离 / Σ参考字数），
  长句权重更高，比"句级平均"更稳健；
- `sentence_accuracy`：归一化后**整句全对**的句子占比。

In [7]:
# ============================================================
# Cell 7：编辑距离与错率实测
# ============================================================
print("edit_distance（替换/删除/插入 各一例）：")
for a, b in [("abc", "abc"), ("abc", "abd"), ("abc", "ab"), ("abc", "abcd"), ("", "ab")]:
    print(f"    edit_distance({a!r}, {b!r}) = {metrics.edit_distance(a, b)}")

print()
print("char_error_rate：")
print("    完全相同           ->", metrics.char_error_rate("你好世界", "你好世界"))
print("    '你好世界' vs '你好' ->", metrics.char_error_rate("你好世界", "你好"))
print("    '619' vs '六幺九'    ->", metrics.char_error_rate("619", "六幺九"), "（归一化后一致，故为 0）")

print()
print("word_error_rate（英文）：")
print("    'vpn error 619' vs 'vpn error 619' ->", metrics.word_error_rate("vpn error 619", "vpn error 619"))
print("    'vpn error 619' vs 'vpn 619'       ->", round(metrics.word_error_rate("vpn error 619", "vpn 619"), 4))

pairs = [("你好世界", "你好"), ("错误619", "错误619")]
print()
print("corpus_cer（语料级，参考总长 4+5=9，编辑距离 2）：", round(metrics.corpus_cer(pairs), 4))
print("sentence_accuracy（2 句对了 1 句）：", metrics.sentence_accuracy(pairs))

edit_distance（替换/删除/插入 各一例）：
    edit_distance('abc', 'abc') = 0
    edit_distance('abc', 'abd') = 1
    edit_distance('abc', 'ab') = 1
    edit_distance('abc', 'abcd') = 1
    edit_distance('', 'ab') = 2

char_error_rate：
    完全相同           -> 0.0
    '你好世界' vs '你好' -> 0.5
    '619' vs '六幺九'    -> 0.0 （归一化后一致，故为 0）

word_error_rate（英文）：
    'vpn error 619' vs 'vpn error 619' -> 0.0
    'vpn error 619' vs 'vpn 619'       -> 0.3333

corpus_cer（语料级，参考总长 4+5=9，编辑距离 2）： 0.2222
sentence_accuracy（2 句对了 1 句）： 0.5


### 4.4 检索指标：四个指标各有分工

| 指标 | 看什么 | 直觉 |
|------|--------|------|
| `recall@k` | 相关文档**找全没有** | 找全了才不遗漏 |
| `hit@k` | 前 k 条**有没有**命中 | 最宽松 |
| `reciprocal_rank` / `mrr` | **第一个命中排多靠前** | 越靠前越好 |
| `ndcg@k` | 命中**数量 + 位置**综合 | 兼顾"有"和"靠前" |

本项目检索套件的主指标是 **多模态模式的 `recall@3`**，
并用同题的 `text_only`（纯文本模式）作对照——这就是"文查图增益"的量化方式。

In [8]:
# ============================================================
# Cell 8：检索排序指标实测
# ============================================================
ranked = ["a", "b", "c"]     # 检索返回的排名
gold = ["b", "d"]            # 金标相关项

print(f"ranked = {ranked}，gold = {gold}")
print()
print(f"    recall@3       = {metrics.recall_at_k(ranked, gold, 3)}   （2 个相关项命中 1 个）")
print(f"    precision@3    = {round(metrics.precision_at_k(ranked, gold, 3), 4)}   （前 3 条命中 1 条）")
print(f"    hit@1          = {metrics.hit_at_k(ranked, gold, 1)}   （第一位是 a，未命中）")
print(f"    hit@2          = {metrics.hit_at_k(ranked, gold, 2)}   （前两位含 b，命中）")
print(f"    reciprocal_rank= {metrics.reciprocal_rank(['x', 'a'], ['a'])}   （第 2 位命中 → 1/2）")
print(f"    ndcg@3         = {metrics.ndcg_at_k(['a', 'b', 'c'], ['a'], 3)}   （理想排序）")
print()
print("    MRR（多条查询的 RR 均值）：", metrics.mrr([["a"], ["x", "b"]], [["a"], ["b"]]))

ranked = ['a', 'b', 'c']，gold = ['b', 'd']

    recall@3       = 0.5   （2 个相关项命中 1 个）
    precision@3    = 0.3333   （前 3 条命中 1 条）
    hit@1          = 0.0   （第一位是 a，未命中）
    hit@2          = 1.0   （前两位含 b，命中）
    reciprocal_rank= 0.5   （第 2 位命中 → 1/2）
    ndcg@3         = 1.0   （理想排序）

    MRR（多条查询的 RR 均值）： 0.75


### 4.5 结构化字段与描述质量

VLM 有两个出口，量纲完全不同：

- **截图诊断** → 结构化字段：`error_code`（精确匹配）、`service`（**宽松子串匹配**，
  因为模型常写"VPN 客户端"而金标是"VPN"）、`error_type`（枚举精确匹配）；
- **配图描述** → 自由文本：用 `key_point_hit_rate`（人工列关键点，命中率为主指标），
  余弦相似度只作初筛。

In [9]:
# ============================================================
# Cell 9：字段匹配 / 描述质量 / 统计工具实测
# ============================================================
print("结构化字段：")
print("    exact_match('619', '错误码 619') ->", metrics.exact_match("619", "错误码 619"), "（归一化后 error619≠619）")
print("    exact_match('619', '619')        ->", metrics.exact_match("619", "619"))
print("    substring_match('VPN 客户端','VPN')->", metrics.substring_match("VPN 客户端", "VPN"), "（宽松匹配）")
print("    substring_match('邮箱客户端','VPN')->", metrics.substring_match("邮箱客户端", "VPN"))
print("    field_accuracy(['619','502'], ['619','504']) ->", metrics.field_accuracy(["619", "502"], ["619", "504"]))

print()
print("描述质量：")
text = "企业网络架构：员工终端经公网接入 VPN 网关与内网应用服务器。"
print("    文本：", text)
print("    key_point_hit_rate(['员工终端','VPN网关'])           ->",
      metrics.key_point_hit_rate(text, ["员工终端", "VPN网关"]))
print("    key_point_hit_rate(['员工终端','VPN网关','防火墙'])  ->",
      round(metrics.key_point_hit_rate(text, ["员工终端", "VPN网关", "防火墙"]), 4))
print("    cosine([1,0],[0,1]) ->", metrics.cosine([1, 0], [0, 1]))

print()
print("统计工具：")
print("    mean([1,2,3,4])          ->", metrics.mean([1, 2, 3, 4]))
print("    percentile([1,2,3,4],0.5)->", metrics.percentile([1, 2, 3, 4], 0.5), "（时延中位数）")
print("    percentile([1,2,3,4],0.95)->", round(metrics.percentile([1, 2, 3, 4], 0.95), 2), "（时延 P95）")
print("    pearson([1,2,3],[2,4,6]) ->", metrics.pearson([1, 2, 3], [2, 4, 6]), "（完全正相关）")

结构化字段：
    exact_match('619', '错误码 619') -> False （归一化后 error619≠619）
    exact_match('619', '619')        -> True
    substring_match('VPN 客户端','VPN')-> True （宽松匹配）
    substring_match('邮箱客户端','VPN')-> False
    field_accuracy(['619','502'], ['619','504']) -> 0.5

描述质量：
    文本： 企业网络架构：员工终端经公网接入 VPN 网关与内网应用服务器。
    key_point_hit_rate(['员工终端','VPN网关'])           -> 1.0
    key_point_hit_rate(['员工终端','VPN网关','防火墙'])  -> 0.6667
    cosine([1,0],[0,1]) -> 0.0

统计工具：
    mean([1,2,3,4])          -> 2.5
    percentile([1,2,3,4],0.5)-> 2.5 （时延中位数）
    percentile([1,2,3,4],0.95)-> 3.85 （时延 P95）
    pearson([1,2,3],[2,4,6]) -> 1.0 （完全正相关）


---
## Part 5：公共流水线 `_common.py` —— 四个套件共用的一套骨架
---

### 5.1 四个套件其实走同一条流水线

```
读用例(load_cases) → 逐条执行(map_limited，限流并发) → 聚合指标 → 判定(verdict)
                                                              → 装配(build_result)
```

把这套共性抽到 `_common.py`，四个套件就不用各写一遍。三个关键点：

1. **`jsonl` 逐行解析，坏行只丢当行**（不影响其余用例）；
2. **并发限流**（`map_limited` + 信号量），既提速又不会把外部 ASR/TTS/VLM 接口打爆；
3. **SKIP 与 FAIL 严格区分**——这是评测可信度的底线。

In [10]:
# ============================================================
# Cell 10：verdict / combine_verdict 判定源码 + 实测
# ============================================================
print("=" * 66)
print("  verdict —— 单指标判定（value=None 时返回 SKIP，而不是 FAIL）")
print("=" * 66)
print(inspect.getsource(common_mod.verdict))
print("=" * 66)
print("  combine_verdict —— 多指标一票否决")
print("=" * 66)
print(inspect.getsource(common_mod.combine_verdict))

print("实测：")
print(f"    verdict(0.10, 0.15, lower_is_better=True)  -> {common_mod.verdict(0.10, 0.15, True)}   （CER 0.10 ≤ 0.15）")
print(f"    verdict(0.20, 0.15, lower_is_better=True)  -> {common_mod.verdict(0.20, 0.15, True)}   （CER 0.20 > 0.15）")
print(f"    verdict(None,  0.15, lower_is_better=True) -> {common_mod.verdict(None, 0.15, True)}   （未测到 → SKIP，不是 FAIL）")
print(f"    combine_verdict(a=0.6 达标, b=0.4 不达标)   -> "
      f"{common_mod.combine_verdict([('a', 0.5, False), ('b', 0.5, False)], {'a': 0.6, 'b': 0.4})}")
print()
print("  为什么 SKIP≠FAIL：把环境问题（依赖没装/知识库没播种）当成质量问题，会误伤被测能力。")

  verdict —— 单指标判定（value=None 时返回 SKIP，而不是 FAIL）
def verdict(value: Optional[float], threshold: float, lower_is_better: bool = False) -> str:
    """单指标 PASS/FAIL 判定；value 为 None（未测）时返回 SKIP。

    参数：
        value：实测指标值；None 表示该指标没有测到（如 ASR 用例全部调用失败）。
        threshold：阈值，来自 ``EVAL_THRESHOLD_*`` 配置（默认值是经验值，跑通后
            按实测分布校准）。
        lower_is_better：True 表示越小越好（如 CER），判定 ``value <= threshold``；
            False 表示越大越好（如准确率 / 召回率），判定 ``value >= threshold``。
    返回：
        "PASS" / "FAIL" / "SKIP" 之一。
    """
    if value is None:
        return "SKIP"  # 未测到不能判 FAIL，否则把环境问题误当质量不达标
    ok = value <= threshold if lower_is_better else value >= threshold
    return "PASS" if ok else "FAIL"

  combine_verdict —— 多指标一票否决
def combine_verdict(criteria: list[tuple[str, float, bool]], metrics: dict) -> str:
    """多指标联合判定：(指标名, 阈值, 越低越好)。

    参数：
        criteria：判定项列表，每项形如 ``(指标名, 阈值, 越低越好)``。例如 VLM 同时
            检查「诊断字段准确率」和「描述关键点命中率」。
        metrics：指标名 → 实测值。
    返回：
        全部通过 

In [11]:
# ============================================================
# Cell 11：用例加载 / 限流 / 结果装配 源码
# ============================================================
print("=" * 66); print("  load_cases —— jsonl 逐行解析，坏行只丢当行"); print("=" * 66)
print(inspect.getsource(common_mod.load_cases))
print("=" * 66); print("  map_limited —— 信号量限流并发"); print("=" * 66)
print(inspect.getsource(common_mod.map_limited))

import json as _json
cases = common_mod.load_cases("retrieval_cases.jsonl")
print(f"实测：加载 retrieval_cases.jsonl 共 {len(cases)} 条，前 2 条：")
for c in cases[:2]:
    print("   ", _json.dumps(c, ensure_ascii=False))

  load_cases —— jsonl 逐行解析，坏行只丢当行
def load_cases(filename: str, limit: Optional[int] = None) -> list[dict]:
    """读取 jsonl 用例（跳过空行），按 limit 截断。

    参数：
        filename：数据集文件名，如 ``speech_cases.jsonl``（相对 DATASETS_DIR）。
        limit：最多返回条数。传 None 时用配置上限 ``EVAL_MAX_CASES``（默认 20，
            是为控制外部 API 成本的小样本限流）；显式传 0 表示一条都不跑。
    返回：
        用例 dict 列表。文件不存在返回空列表；某行非法 JSON 则跳过该行并告警，
        不影响其余用例（这正是 jsonl 逐行解析的容错优势）。
    """
    path = DATASETS_DIR / filename
    if not path.exists():
        return []
    cases: list[dict] = []
    # 逐行解析：每行是一个独立的 JSON 对象
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:  # 跳过空行（文件末尾/分隔用）
            continue
        try:
            cases.append(json.loads(line))
        except json.JSONDecodeError:
            logger.warning("用例解析失败，已跳过：%s", line[:80])
    # limit=None 用配置上限；显式传 0 表示一条都不跑（不能用真假值混同二者）
    size = settings.EVAL_MAX_CASES if limit is None else limit
    return cases[:siz

> 注意 `load_cases` 里有一行很讲究的注释：**`limit=None`（用默认上限）
> 与 `limit=0`（一条都不跑）是两种不同语义**，不能用真假值混同——
> 所以写的是 `settings.EVAL_MAX_CASES if limit is None else limit`。
> 这类"边界语义"正是评测代码最容易写错的地方。

### 5.2 结果装配：所有套件 return 同一种结构

`build_result` 保证每个套件返回统一结构（`sample_count / metrics / items /
criteria / verdict / notes`），runner 才能无差别落库、前端才能无差别渲染。
跑不起来时用 `skipped(...)` 返回 `verdict="SKIP"` 并写明原因。

In [12]:
# ============================================================
# Cell 12：统一结果结构（含 skipped 占位）
# ============================================================
print("=" * 66); print("  build_result —— 统一装配"); print("=" * 66)
print(inspect.getsource(common_mod.build_result))
print("=" * 66); print("  skipped —— 无法执行时的 SKIP 占位"); print("=" * 66)
print(inspect.getsource(common_mod.skipped))

demo = common_mod.skipped("retrieval", "检索（多模态 vs 纯文本）",
                          "演示知识库未就绪，缺少金标来源：vpn.md")
print("实测 skipped 的返回：")
print(f"    verdict={demo['verdict']}  sample_count={demo['sample_count']}")
print(f"    notes={demo['notes']}")
print("  → 报告页展示'请先做前置操作'，而不是抛错。")

  build_result —— 统一装配
def build_result(
    suite: str,
    title: str,
    metrics: dict,
    items: list[dict],
    primary: Optional[str] = None,
    primary_threshold: float = 0.0,
    lower_is_better: bool = False,
    criteria: Optional[list[tuple[str, float, bool]]] = None,
    notes: Optional[list[str]] = None,
) -> dict:
    """装配套件结果（所有套件 return 的统一结构）。

    参数：
        suite / title：套件标识与展示标题。
        metrics：指标名 → 实测值（含 null 表示该指标未测）。
        items：逐条明细（前端报告页可展开查看每条）。
        primary / primary_threshold / lower_is_better：单指标判定（简单套件用）；
        criteria：多指标联合判定（如 VLM 同时看诊断准确率与描述命中率）。
        notes：备注（如跳过的套件及原因、样本局限说明）。
    返回：
        统一结果 dict，含 ``sample_count`` / ``metrics`` / ``items`` / ``criteria``
        / ``verdict``（PASS/FAIL/SKIP）/ ``notes``。runner 会把它整体落库。
    """
    if criteria:
        # 多指标：任一不达标则 FAIL，全未测则 SKIP
        verdict_text = combine_verdict(criteria, metrics)
    elif primary:
        # 单指标：直接对该主指标做判定
        verdict_text = verdict(metrics.get(primar

---
## Part 6：四个套件走读
---

### 6.1 ASR 套件：用 TTS "自建主集"，零人工录音

ASR 评测最难的是**没有金标音频**。本项目的巧解是：
**用 TTS 现场合成音频 → 再用 ASR 转写 → 与原文比算 CER**，因此不需要人工录音。

| 指标 | 口径 | 方向 |
|------|------|------|
| `cer`（主指标） | 语料级字错率 = Σ编辑距离 / Σ参考字数 | **越低越好**，阈值 0.15 |
| `sentence_accuracy` | 整句全对的句子占比 | 越高越好 |
| `failure_rate` | 真正跑了却失败的占比 | 越低越好 |

> ⚠️ **必须如实标注的局限**：TTS 与 ASR 是**异厂商**（edge-tts vs api），
> 复合误差可作为识别准确率的**合理代理**；且自建音频无口音/噪声/语速变化，
> **CER 属乐观下界**——报告里要写清楚，不能当成真实场景的绝对准确率。

In [13]:
# ============================================================
# Cell 13：ASR 套件源码走读
# ============================================================
import eval.suites.asr as asr_suite

print(f"SUITE={asr_suite.SUITE!r}  TITLE={asr_suite.TITLE!r}")
print("=" * 66); print("  _one —— 单条：TTS 合成 → ASR 转写 → 算 CER"); print("=" * 66)
print(inspect.getsource(asr_suite._one))
print("要点：")
print("  - 合成失败 → synth_failed=True，该条不计入 CER 分母（否则'没测'会被算成'全错'）")
print("  - 转写失败 → hypothesis 加 FAIL_PREFIX 前缀，cer 留 None（未测）")

SUITE='asr'  TITLE='语音识别（ASR）字错率'
  _one —— 单条：TTS 合成 → ASR 转写 → 算 CER
async def _one(case: dict) -> dict:
    """跑单条 ASR 用例：TTS 合成 → ASR 转写 → 算 CER。

    参数：
        case：一条 jsonl 用例，含 ``id`` / ``text``（参考文本）/ ``voice``（可选音色）。
    返回：
        明细 dict，含 ``hypothesis``（ASR 转写结果）、``cer``（句级字错率，未测为
        None）、``latency_ms``（合成+转写总耗时）等。
    异常：
        不向外抛异常。合成失败时置 ``synth_failed=True`` 并返回（该条不计入 CER
        分母）；转写失败时把 ``hypothesis`` 置为 FAIL_PREFIX 前缀串，让上层识别为
        「未测」而非「识别错误」。
    """
    voice = case.get("voice") or settings.TTS_VOICE  # 用例未指定音色则用全局默认
    started = time.perf_counter()  # 计时起点，覆盖合成+转写全流程
    try:
        audio, ext = await synthesize_speech(case.get("text", ""), voice)
    except Exception as exc:  # noqa: BLE001
        # 合成失败 ⇒ 根本没有音频可测，单独标记 synth_failed，不计入 CER 分母
        return {
            "id": case.get("id"),
            "text": case.get("text"),
            "voice": voice,
            "ok": False,
            "synth_failed": True,
            "hypothesis":

### 6.2 TTS 套件：没有"正确音频"怎么评？

TTS 没有唯一正确输出，所以用三类**可自动测量**的代理指标：

1. **反向 ASR CER**（主指标）：合成音频再喂 ASR，与原文比对 → **可懂度**代理；
2. **副语言指标**：合成成功率、时延 P50/P95、平均语速（字/秒）；
3. **时长一致性**：音频时长 vs 文本长度的皮尔逊相关，正常应接近 1
   （相关太低 ⇒ 疑似漏读/截断）。

有个工程巧思：**用纯 Python 解析 MP3 帧头估时长**，从而**免掉 ffmpeg 依赖**。

In [14]:
# ============================================================
# Cell 14：TTS 套件源码走读
# ============================================================
import eval.suites.tts as tts_suite

print(f"SUITE={tts_suite.SUITE!r}  TITLE={tts_suite.TITLE!r}")
print("=" * 66); print("  _mp3_duration_sec —— 纯 Python 解析 MP3 帧头估时长"); print("=" * 66)
print(inspect.getsource(tts_suite._mp3_duration_sec))
print("=" * 66); print("  run —— 聚合副语言指标"); print("=" * 66)
print(inspect.getsource(tts_suite.run))
print("要点：")
print("  - 语速超过 _MAX_REASONABLE_CPS=8 字/秒 → 计入 truncation_rate（疑似漏读）")
print("  - duration_len_corr 用 pearson(文本长度, 时长)，偏低说明合成不稳定")

SUITE='tts'  TITLE='语音合成（TTS）可懂度与副语言'
  _mp3_duration_sec —— 纯 Python 解析 MP3 帧头估时长
def _mp3_duration_sec(data: bytes) -> float:
    """估算 MP3 时长（秒）。逐帧按「每帧样本数 / 采样率」累加。

    参数：
        data：MP3 文件字节。
    返回：
        估算时长（秒）；空数据或解析不到有效帧时返回 0.0。
    说明：
        纯 Python 解析 MP3 帧头，避免引入 ffmpeg 依赖：先跳过 ID3v2 标签，再逐帧
        读版本/层/比特率/采样率，算出每帧时长并累加。
    """
    if not data:
        return 0.0
    index = 0
    size = len(data)
    if data[:3] == b"ID3":  # 跳过 ID3v2 标签，否则会把标签字节误当音频帧
        tag_size = (
            ((data[6] & 0x7F) << 21)
            | ((data[7] & 0x7F) << 14)
            | ((data[8] & 0x7F) << 7)
            | (data[9] & 0x7F)
        )
        index = 10 + tag_size

    total = 0.0
    while index + 4 <= size:
        # 同步字 0xFFE：非帧头则右移一字节继续找
        if data[index] != 0xFF or (data[index + 1] & 0xE0) != 0xE0:
            index += 1
            continue
        version = (data[index + 1] >> 3) & 0x03   # 3=MPEG1, 2=MPEG2, 0=MPEG2.5
        layer = (data[index + 1] >> 1) & 0x0

### 6.3 VLM 套件：两类出口，分开评、联合判

VLM 在本项目有两个量纲完全不同的出口，所以**分开评、最后联合判定（一票否决）**：

| 出口 | 评测方式 | 主指标 |
|------|----------|--------|
| **截图诊断** `process_image()` | 结构化字段逐字段比对 | `diagnosis_field_accuracy` |
| **配图描述** `describe_knowledge_image()` | 关键点命中率 + 余弦初筛 + 幻觉检测 | `caption_keypoint_hit` |

幻觉检测由 `judges.py` 的 **LLM-as-judge** 完成。

> ⚠️ 局限：judge 与生成**同源**（同一个 VLM）时存在同源偏差，报告需如实标注；
> judge 输出解析失败时按 `available=False` **跳过**，绝不臆断为"无幻觉"。

In [15]:
# ============================================================
# Cell 15：VLM 套件源码走读
# ============================================================
import eval.suites.vlm as vlm_suite

print(f"SUITE={vlm_suite.SUITE!r}  TITLE={vlm_suite.TITLE!r}")
print("=" * 66); print("  _diag_one —— 截图诊断：逐字段比对"); print("=" * 66)
print(inspect.getsource(vlm_suite._diag_one))
print("=" * 66); print("  run —— 多指标联合判定（criteria）"); print("=" * 66)
print(inspect.getsource(vlm_suite.run))
print("要点：")
print("  - service 用 substring_match（宽松）：gold='VPN'、pred='VPN 客户端' 也算命中")
print("  - criteria 两项都达标才 PASS（一票否决），避免高分项掩盖短板")

SUITE='vlm'  TITLE='图文理解（截图诊断 + 配图描述）'
  _diag_one —— 截图诊断：逐字段比对
async def _diag_one(case: dict) -> dict:
    """跑单条「截图诊断」用例：取图 → VLM 结构化诊断 → 逐字段比对金标。

    参数：
        case：用例，含 ``id`` / ``image``（图片文件名）/ ``gold``（金标字段 dict）。
    返回：
        明细 dict：``pred`` 预测字段、``matches`` 各字段是否命中、``field_hit``
        （三字段命中率）、``ok`` 等。
    异常：
        不向外抛异常；取图或诊断失败时返回带 error 的 ``ok=False`` 明细。
    """
    gold = case.get("gold") or {}
    try:
        image = materialize(case.get("image", ""))  # 按文件名取或即时生成合成图
        pred = await process_image(image)
    except Exception as exc:  # noqa: BLE001
        return {"kind": "diagnosis", "id": case.get("id"), "image": case.get("image"),
                "gold": gold, "pred": {}, "matches": {}, "ok": False, "error": str(exc)}

    # 分字段比对：error_code/error_type 用归一化精确匹配；service 用子串宽松匹配
    # （模型常带修饰词，如 gold="VPN"、pred="VPN 客户端"，严格相等会误判）
    matches = {
        "error_code": metrics.exact_match(pred.get("error_code"), gold.get("error_code")),
        "servi

### 6.4 检索套件：同一 query 跑两种模式做对照

这是最能体现"多模态 RAG 价值"的套件：对每个 query，**分别跑多模态模式与
`text_only` 纯文本模式**，各自算召回指标，最后给出增益差 `multimodal_gain@3`。

还有一个很务实的前置检查：**金标来源必须都在已索引文档里**，
否则检索必然找不中——这属于"环境没准备好"，返回 `SKIP` 并提示先加载演示知识库，
**不误报为 FAIL**。

In [16]:
# ============================================================
# Cell 16：检索套件源码走读
# ============================================================
import eval.suites.retrieval as retrieval_suite

print(f"SUITE={retrieval_suite.SUITE!r}  TITLE={retrieval_suite.TITLE!r}")
print("=" * 66); print("  _rank —— 检索后按来源去重，保留首次出现顺序"); print("=" * 66)
print(inspect.getsource(retrieval_suite._rank))
print("=" * 66); print("  run —— 前置检查 + 两模式对照 + 增益差"); print("=" * 66)
print(inspect.getsource(retrieval_suite.run))
print("要点：")
print("  - 按来源文件去重：同一来源只保留排名最高的一次，避免同文件多条挤占排名")
print("  - 缺金标来源 → skipped(...) 返回 SKIP，而不是 FAIL")

SUITE='retrieval'  TITLE='检索（多模态 vs 纯文本）'
  _rank —— 检索后按来源去重，保留首次出现顺序
async def _rank(query: str, text_only: bool) -> list[str]:
    """检索并按来源文件去重，保留首次出现的顺序作为排名。

    参数：
        query：查询文本。
        text_only：True 走纯文本检索，False 走多模态检索（二者互为对照）。
    返回：
        去重后的来源文件名列表，按检索得分从高到低排列（即排名）。
    """
    results = await multimodal_rag.search(query=query, text_only=text_only)
    ranked: list[str] = []
    for item in results:
        source = (item.get("metadata") or {}).get("source")
        if source and source not in ranked:  # 同一来源只保留排名最高的一次
            ranked.append(source)
    return ranked

  run —— 前置检查 + 两模式对照 + 增益差
async def run(limit: int | None = None) -> dict:
    """执行检索套件：多模态 vs 纯文本对照评测。

    参数：
        limit：最多评测条数；None 时用 ``EVAL_MAX_CASES``。
    返回：
        统一结果 dict；若知识库未就绪则返回 SKIP 占位结果（不误报为 FAIL）。
    """
    cases = load_cases("retrieval_cases.jsonl", limit)
    if not cases:
        return build_result(SUITE, TITLE, {}, [], notes=["无可用用例（datasets/retrieval_cases.jsonl

---
## Part 7：数据与素材 —— jsonl 用例 + 确定性合成图
---

### 7.1 用例格式：为什么用 jsonl

`eval/datasets/*.jsonl` 每行一个独立 JSON 对象。相比一个大 JSON 数组：

- **能逐行流式解析**，不用一次读进内存；
- **某行坏了只丢那一行**，不影响其余用例（`load_cases` 里 catch 的就是这个）；
- 追加新用例只需在末尾加一行，不会破坏全文。

In [17]:
# ============================================================
# Cell 17：用例文件总览 + 检索用例样例
# ============================================================
from collections import Counter

print("=" * 66)
print("  eval/datasets/ 用例文件")
print("=" * 66)
for p in sorted(Path("eval/datasets").glob("*.jsonl")):
    n = len([l for l in p.read_text(encoding="utf-8").splitlines() if l.strip()])
    print(f"    {p.name:<26} {n:>3} 条")

print()
print("=" * 66)
print("  retrieval_cases.jsonl 样例（query → 金标来源）")
print("=" * 66)
for c in common_mod.load_cases("retrieval_cases.jsonl")[:5]:
    print(f"    {c['id']}: {c['query'][:34]:<36} gold={c['gold']}")
print()
print("  注意 ret-007 / ret-010 的 gold 里含 .png —— 正是'文查图'的评测点。")

  eval/datasets/ 用例文件
    caption_cases.jsonl          3 条
    retrieval_cases.jsonl       10 条
    speech_cases.jsonl          10 条
    vlm_cases.jsonl              3 条

  retrieval_cases.jsonl 样例（query → 金标来源）
    ret-001: VPN 连接失败提示错误 619 怎么解决                gold=['vpn.md']
    ret-002: 忘记密码了如何重置企业账号密码                      gold=['password.md']
    ret-003: 企业邮箱 IMAP 和 SMTP 服务器端口怎么配置           gold=['email.md']
    ret-004: 安装软件时提示没有管理员权限怎么办                    gold=['software.md']
    ret-005: Nginx 502 Bad Gateway 是什么原因导致的       gold=['error_code_table.md']

  注意 ret-007 / ret-010 的 gold 里含 .png —— 正是'文查图'的评测点。


### 7.2 评测素材：确定性合成图（不依赖任何外部素材）

评测要图，但不能用真实数据（隐私问题）。`images.py` 用 PIL **画出**
"报错截图 / 拓扑图"，**确定性生成**（同一次输入永远得到同一张图），
金标写在 jsonl 里。生成结果缓存到 `data/eval/images/`。

In [18]:
# ============================================================
# Cell 18：合成图实测（内存中绘制，不写磁盘）
# ============================================================
print("=" * 66)
print("  images.BUILDERS —— 文件名 → 生成函数")
print("=" * 66)
for name, fn in images_mod.BUILDERS.items():
    print(f"    {name:<22} -> {fn.__name__}()  {fn.__doc__.strip()}")

from io import BytesIO
from PIL import Image

data = images_mod.vpn_619()          # 内存中绘制，返回 PNG 字节
img = Image.open(BytesIO(data))
print()
print("实测 vpn_619()：")
print(f"    PNG 字节数 = {len(data)}  尺寸 = {img.size}")
print("  → 合成图确定性生成、无真实个人信息，评测不依赖外部素材。")

  images.BUILDERS —— 文件名 → 生成函数
    vpn_619.png            -> vpn_619()  VPN 错误 619 报错截图。金标：error_code=619 / service=VPN / error_type=网络。
    nginx_502.png          -> nginx_502()  Nginx 502 Bad Gateway 报错页。金标：error_code=502 / service=Nginx / error_type=网络。
    vpn_691.png            -> vpn_691()  VPN 错误 691（安全验证失败）报错截图。金标：error_code=691 / service=VPN / error_type=权限。
    network_topology.png   -> network_topology()  企业网络与 VPN 接入路径拓扑图（用于配图描述评测）。



实测 vpn_619()：
    PNG 字节数 = 33643  尺寸 = (760, 360)
  → 合成图确定性生成、无真实个人信息，评测不依赖外部素材。


---
## Part 8：编排与持久化 —— runner + store
---

### 8.1 runner：一次评测怎么被组织起来

生命周期只有三步，关键是**"接口立即返回、执行放后台"**：

```
create_run(suites)   建记录（status=running）→ 立即返回 run_id 给前端
execute(run_id)      后台逐套件跑，每跑完一个就落盘（前端轮询看进度）
finalize             收尾置 done/failed
```

两个健壮性设计值得学：

1. **单套件失败只降级为 FAIL**，不连累其他套件继续跑；
2. **被取消（服务重启）时显式收尾**——`asyncio.CancelledError` 不会被
   `except Exception` 捕获，必须单独处理，否则记录会永久停在 `running` 且删不掉。

In [19]:
# ============================================================
# Cell 19：runner 源码走读（配置快照 + 执行编排）
# ============================================================
print("=" * 66); print("  _config_snapshot —— 记录模型与阈值，保证结果可复现"); print("=" * 66)
print(inspect.getsource(runner_mod._config_snapshot))
print("=" * 66); print("  create_run —— 先落盘再返回，执行交后台"); print("=" * 66)
print(inspect.getsource(runner_mod.create_run))
print("=" * 66); print("  execute —— 逐套件执行 + 失败降级 + 取消收尾"); print("=" * 66)
print(inspect.getsource(runner_mod.execute))

  _config_snapshot —— 记录模型与阈值，保证结果可复现
def _config_snapshot() -> dict:
    """记录本次评测的模型与阈值，保证结果可复现、可溯源。

    返回：
        快照 dict：各模态模型名、检索参数、样本上限与并发、以及各套件阈值。
        历史报告带上它，才能解释「为什么这次的分数和上次不一样」。
    """
    return {
        "asr_provider": settings.ASR_PROVIDER,
        "asr_model": settings.ASR_MODEL,
        "tts_provider": settings.TTS_PROVIDER,
        "tts_voice": settings.TTS_VOICE,
        "vlm_model": settings.VLM_MODEL,
        "embedding_model": settings.EMBEDDING_MODEL,
        "reranker_model": settings.RERANKER_MODEL,
        "retrieval_top_k": settings.RETRIEVAL_TOP_K,
        "reranker_top_n": settings.RERANKER_TOP_N,
        "max_cases": settings.EVAL_MAX_CASES,
        "concurrency": settings.EVAL_CONCURRENCY,
        "thresholds": {
            "asr_cer": settings.EVAL_THRESHOLD_ASR_CER,
            "tts_cer": settings.EVAL_THRESHOLD_TTS_CER,
            "vlm_field_acc": settings.EVAL_THRESHOLD_VLM_FIELD_ACC,
            "caption_keypoint": settings.EVAL_THRESHOLD_CAP

### 8.2 store：文件式持久化，零额外依赖

结果直接落成 `data/eval/runs/{run_id}.json`，基线落 `data/eval/baseline.json`。
三个安全/健壮细节：

| 细节 | 为什么 |
|------|--------|
| **run_id 白名单校验** | run_id 会拼进文件名，不校验就能用 `../../` 路径穿越 |
| **原子写**（临时文件 + `os.replace`） | 边跑边落盘 + 前端 2s 轮询，非原子写会读到"写了一半"的 JSON |
| **`recover_stale_runs()`** | 进程被杀时 run 停在 running，启动时统一收敛为 failed，否则永远删不掉 |

In [20]:
# ============================================================
# Cell 20：store 源码走读 + 真跑（临时目录，不污染 data/eval）
# ============================================================
import tempfile

print("=" * 66); print("  _run_path —— run_id 白名单防路径穿越"); print("=" * 66)
print(inspect.getsource(store_mod._run_path))
print("=" * 66); print("  _atomic_write_text —— 原子写"); print("=" * 66)
print(inspect.getsource(store_mod._atomic_write_text))
print("=" * 66); print("  recover_stale_runs —— 收敛遗留 running"); print("=" * 66)
print(inspect.getsource(store_mod.recover_stale_runs))

print("实测（把 store 的目录指到临时目录）：")
print(f"    new_run_id() 形如：{store_mod.new_run_id()}")
try:
    store_mod._run_path("../../etc/passwd")
except ValueError as exc:
    print(f"    非法 run_id -> ValueError：{exc}")

tmp = Path(tempfile.mkdtemp())
store_mod.RUNS_DIR = tmp / "runs"
store_mod.BASELINE_PATH = tmp / "baseline.json"
run = {"id": store_mod.new_run_id(), "status": "running", "suites": ["asr"],
       "started_at": store_mod.now_iso(), "finished_at": "",
       "progress": {"done": 0, "total": 1, "current": ""},
       "results": {}, "notes": [], "error": ""}
store_mod.save_run(run)
print(f"    save → load 一致：{store_mod.load_run(run['id'])['id'] == run['id']}")
print(f"    list_runs 条数：{len(store_mod.list_runs())}")
print(f"    recover_stale_runs() 恢复 {store_mod.recover_stale_runs()} 条，"
      f"状态变为 {store_mod.load_run(run['id'])['status']}")

  _run_path —— run_id 白名单防路径穿越
def _run_path(run_id: str) -> Path:
    """由 run_id 解析出结果文件路径，并做白名单校验。

    参数：
        run_id：运行 id。
    返回：
        对应的 JSON 文件路径。
    异常：
        ValueError：run_id 不符合约定格式时抛出（防止路径穿越）。
    """
    if not _ID_RE.match(run_id or ""):
        raise ValueError(f"非法 run_id：{run_id}")
    return RUNS_DIR / f"{run_id}.json"

  _atomic_write_text —— 原子写
def _atomic_write_text(path: Path, text: str) -> None:
    """同目录临时文件 + 原子替换，避免读侧读到「写了一半」的 JSON。

    runner 边跑边落盘、前端每 2s 轮询，非原子写会偶发半截内容导致
    详情 404 或列表漏项。
    """
    tmp = path.with_name(f"{path.name}.{uuid.uuid4().hex[:6]}.tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

  recover_stale_runs —— 收敛遗留 running
def recover_stale_runs() -> int:
    """把遗留在 ``running`` 的记录标记为 ``failed``（进程重启后调用）。

    后台任务随进程被杀时不会走到收尾逻辑，run 会永久停在 running，
    而删除接口又禁止删 running——该记录将永远无法清理。启动时统一收敛。
    返回被恢复的记录数。
    """
    if not RUNS_DIR.exists():
        return 0
    recovered = 0
    for path in RUNS

---
## Part 9：管理端接口 + 回归测试（简）
---

### 9.1 管理端评测接口

评测由 **admin 接口触发**（不提供命令行入口），前端"评测"页据此一键运行。

In [21]:
# ============================================================
# Cell 21：admin_api.py 中的评测端点清单
# ============================================================
tree = ast.parse(Path("core/api/admin_api.py").read_text(encoding="utf-8"))
print("=" * 66)
print("  /api/admin/eval/* 端点")
print("=" * 66)
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") in ("get", "post", "delete"):
                route = dec.args[0].value if dec.args else "?"
                if "eval" in str(route):
                    print(f"  {dec.func.attr.upper():<7}{route:<34}{node.name}")

print()
print("关键约定：")
print("  - POST /eval/run：建记录后立即返回 run_id，执行放后台任务（asyncio.create_task）")
print("  - DELETE /eval/runs/{id}：running 的记录不可删（避免删了还在写的文件）")
print("  - POST /eval/runs/{id}/baseline：仅 done 的记录可设为基线")

  /api/admin/eval/* 端点
  GET    /eval/suites                      eval_suites
  POST   /eval/run                         start_eval
  GET    /eval/runs                        list_eval_runs
  GET    /eval/runs/{run_id}               eval_run_detail
  DELETE /eval/runs/{run_id}               delete_eval_run
  POST   /eval/runs/{run_id}/baseline      set_eval_baseline

关键约定：
  - POST /eval/run：建记录后立即返回 run_id，执行放后台任务（asyncio.create_task）
  - DELETE /eval/runs/{id}：running 的记录不可删（避免删了还在写的文件）
  - POST /eval/runs/{id}/baseline：仅 done 的记录可设为基线


### 9.2 回归测试（一带而过）

`tests/` 是交付门槛，但**本节不逐用例走读**，只记住三条纪律：

| 纪律 | 说明 |
|------|------|
| **纯函数优先** | 指标库（`metrics.py`）无 IO/无网络，可完全离线单测 |
| **外部依赖用假对象** | ASR/TTS/MCP/WS 用 `monkeypatch` 注入替身，测试不联网、不依赖真实服务 |
| **把"部署一致性"也固化成测试** | 如 `test_dockerfile.py` 断言"core 镜像必须拷 eval 包" |

运行方式：仓库根执行 `pytest -q`，全绿即通过。

In [22]:
# ============================================================
# Cell 22：测试文件与用例数总览（静态统计）
# ============================================================
print("=" * 66)
print("  tests/ 测试文件与用例数（按 test_ 前缀静态统计）")
print("=" * 66)
total = 0
for p in sorted(Path("tests").glob("test_*.py")):
    tree = ast.parse(p.read_text(encoding="utf-8"))
    n = sum(1 for node in ast.walk(tree)
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name.startswith("test_"))
    total += n
    print(f"    {p.name:<26} {n:>3} 例")
print(f"    {'合计':<26} {total:>3} 例")
print()
print("  说明：这里按源码静态统计；若含 pytest 参数化用例，实际以")
print("        `pytest --collect-only -q` 的输出为准。")
print("        交付只认一条：`pytest -q` 全绿。")

print()
print("=" * 66)
print("  test_dockerfile.py 源码（把'部署一致性'固化成测试）")
print("=" * 66)
print(Path("tests/test_dockerfile.py").read_text(encoding="utf-8"))

  tests/ 测试文件与用例数（按 test_ 前缀静态统计）
    test_dockerfile.py           6 例
    test_eval_metrics.py        16 例
    test_handoff.py             12 例
    test_mcp_server.py          11 例
    test_multimodal_asr.py       4 例
    test_rag.py                 24 例
    test_session_media.py        4 例
    合计                          77 例

  说明：这里按源码静态统计；若含 pytest 参数化用例，实际以
        `pytest --collect-only -q` 的输出为准。
        交付只认一条：`pytest -q` 全绿。

  test_dockerfile.py 源码（把'部署一致性'固化成测试）
"""部署一致性测试：deploy/ 下的镜像与编排必须与运行时导入保持一致。

回归背景：
1. ``core.main`` / ``core.api.admin_api`` 在模块顶层导入 ``eval``，但
   Dockerfile 曾漏拷 ``eval``，导致容器内 ``uvicorn core.main:app`` 启动即
   ``ModuleNotFoundError: No module named 'eval'``。
2. 容器部署文件统一收敛到 ``deploy/``，编排里引用的 Dockerfile 必须真实存在。
"""
from __future__ import annotations

from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def test_dockerfile_copies_eval_package():
    text = (ROOT / "deploy" / "Dockerfile.core").read_text(encoding="utf-8")
    assert "CO

---
## Part 10：容器化部署与交付
---

### 10.1 两个镜像 + 一份编排

```
deploy/Dockerfile.core      →  mde-lab-core:1.0.0      （FastAPI + RAG + MCP，:8000）
deploy/Dockerfile.frontend  →  mde-lab-frontend:1.0.0  （Vue 构建产物，:3000）
deploy/docker-compose.yml   →  一键起两个服务 + data 卷持久化
```

三个必须理解的设计点：

1. **构建上下文 = 仓库根**（不是 `deploy/` 或 `frontend/`），所以 Dockerfile 里
   拷贝路径带 `core/`、`frontend/` 前缀；
2. **core 单实例、单 worker**——因为 ChromaDB 嵌入式是**单进程独占**，
   开多 worker 会冲突；
3. **所有持久化数据落 `data` 卷**（`DB_PATH` / `CHROMA_PATH` / `MEDIA_DIR` /
   `TICKETS_DB_PATH`），容器重建数据不丢。

In [23]:
# ============================================================
# Cell 23：Dockerfile.core 走读
# ============================================================
print(Path("deploy/Dockerfile.core").read_text(encoding="utf-8"))
print("要点：")
print("  - 先 COPY requirements.txt 再 pip install，复用镜像层缓存")
print("  - COPY core / eval / mcp_servers：这三者是运行必需（core.main 顶层导入 eval）")
print("  - 不 COPY tests：运行镜像不需要，且 docs/素材被 .dockerignore 排除后测试跑不通")
print("  - HEALTHCHECK 打 /api/health；CMD 单 worker（ChromaDB 嵌入式独占）")

# MDE-LAB 中台（FastAPI + CrewAI + 多模态 RAG），见设计文档 18.1
# 构建上下文 = 仓库根目录：docker compose -f deploy/docker-compose.yml build
# 注意：ChromaDB 为嵌入式（PersistentClient），core 必须单实例运行，不启用多 worker。
FROM python:3.11-slim

ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1 \
    TZ=Asia/Shanghai

WORKDIR /app

# 系统依赖：curl 供健康检查；libgl/字体供 Pillow 渲染中文截图
RUN apt-get update && apt-get install -y --no-install-recommends \
        curl \
        libgl1 \
        libglib2.0-0 \
        fonts-noto-cjk \
    && rm -rf /var/lib/apt/lists/*

# 先装依赖以复用镜像层缓存
COPY requirements.txt ./
RUN pip install --upgrade pip && pip install -r requirements.txt

# 拷贝源码（eval 为 core.main 顶层导入，必需；tests 不随运行镜像分发）
COPY core ./core
COPY eval ./eval
COPY mcp_servers ./mcp_servers

# 运行时数据目录（挂载 ./data 卷，保证容器重建后数据保留）
RUN mkdir -p /app/data/knowledge /app/data/chroma /app/data/media

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=40s --retries=3 \
    CMD curl -fsS http://localhost:8000/api/h

In [24]:
# ============================================================
# Cell 24：Dockerfile.frontend 走读
# ============================================================
print(Path("deploy/Dockerfile.frontend").read_text(encoding="utf-8"))
print("要点：")
print("  - 带 lockfile 时用 npm ci（而非 npm install），保证构建可复现")
print("  - vue 构建产物用 vite preview 提供，靠 preview.proxy 把 /api、/ws 代理到 core")
print("  - BACKEND_TARGET 环境变量指向容器服务名 http://core:8000")

# MDE-LAB 前端（Vue 3 + Vite SPA），见设计文档 18.1
# 构建上下文 = 仓库根目录（与 core 镜像共用，便于在 deploy/ 下统一编排），
# 因此源码路径均以 frontend/ 为前缀，而非在 frontend/ 目录内构建。
FROM node:20-alpine

WORKDIR /app

# 先装依赖以复用镜像层缓存（带 lockfile 用 npm ci，保证构建可复现）
COPY frontend/package.json frontend/package-lock.json ./
RUN npm ci

# 拷贝源码并构建
COPY frontend/ ./
RUN npm run build

EXPOSE 3000

# 用 vite preview 提供构建产物，并按 vite.config.js 的 preview.proxy 代理 /api 与 /ws
CMD ["npm", "run", "preview"]

要点：
  - 带 lockfile 时用 npm ci（而非 npm install），保证构建可复现
  - vue 构建产物用 vite preview 提供，靠 preview.proxy 把 /api、/ws 代理到 core
  - BACKEND_TARGET 环境变量指向容器服务名 http://core:8000


In [25]:
# ============================================================
# Cell 25：docker-compose.yml 走读
# ============================================================
print(Path("deploy/docker-compose.yml").read_text(encoding="utf-8"))
print("要点：")
print("  - volumes: ../data:/app/data —— 一个卷装下向量库/媒体/业务库/工单库")
print("  - environment 把 DB_PATH/CHROMA_PATH/MEDIA_DIR/TICKETS_DB_PATH 都指到卷内")
print("  - env_file 读 ../.env（统一配置，见第 34 节的能力四元组）")
print("  - core 不设多 worker；frontend depends_on core，通过服务名访问")

# MDE-LAB 多模态数字员工平台 —— 部署编排，见设计文档 18.1
#
# 与 Qdrant 方案相比不再需要独立的向量库容器：ChromaDB 以嵌入式库形式
# 随 core 进程运行，数据落在 ../data/chroma。
#
# 用法（在仓库根目录执行）：
#   docker compose -f deploy/docker-compose.yml up -d

services:
  core:
    build:
      # 上下文为仓库根目录：core 镜像需拷贝 core / eval / mcp_servers 等
      context: ..
      dockerfile: deploy/Dockerfile.core
    image: mde-lab-core:1.0.0
    container_name: mde-lab-core
    ports:
      - "8000:8000"
    volumes:
      # 知识库文件、ChromaDB、媒体文件、业务库均持久化在此卷内
      - ../data:/app/data
    env_file:
      - ../.env                   # 统一读取配置（见十七章）
    environment:
      # 工单库与向量库落在数据卷内，容器重启后数据保留
      TICKETS_DB_PATH: /app/data/tickets.db
      DB_PATH: /app/data/app.db
      CHROMA_PATH: /app/data/chroma
      MEDIA_DIR: /app/data/media
    restart: unless-stopped
    # 注意：ChromaDB 为嵌入式，core 必须单实例运行，不启用多 worker

  frontend:
    build:
      # 上下文同为仓库根目录，Dockerfile 内以 frontend/ 前缀拷贝源码
      context: ..
      dockerfile: deploy/Dockerfile.frontend
    image: mde-lab

### 10.2 部署一致性自检：把"文件约定"用代码钉住

镜像与编排之间有很多**跨文件约定**（拷哪些包、卷挂到哪、忽略哪些文件）。
它们最容易在改动中悄悄失效。下面用代码做一次自检——这正是 `test_dockerfile.py`
做的事情。

In [26]:
# ============================================================
# Cell 26：部署一致性自检（真跑）
# ============================================================
core_df = Path("deploy/Dockerfile.core").read_text(encoding="utf-8")
fe_df = Path("deploy/Dockerfile.frontend").read_text(encoding="utf-8")
compose = Path("deploy/docker-compose.yml").read_text(encoding="utf-8")
main_py = Path("core/main.py").read_text(encoding="utf-8")
di = Path(".dockerignore").read_text(encoding="utf-8")
di_patterns = {l.strip() for l in di.splitlines() if l.strip() and not l.startswith("#")}

checks = [
    ("core 镜像拷了 eval 包", "COPY eval ./eval" in core_df),
    ("core.main 顶层依赖 eval", "from eval.store import recover_stale_runs" in main_py),
    ("core 镜像不拷 tests", "COPY tests" not in core_df and "pytest.ini" not in core_df),
    ("frontend 用 npm ci", "npm ci" in fe_df),
    ("frontend 源码路径带前缀", "COPY frontend/ ./" in fe_df),
    ("compose 引用两个 Dockerfile",
     "deploy/Dockerfile.core" in compose and "deploy/Dockerfile.frontend" in compose),
    ("compose 挂 data 卷", "../data:/app/data" in compose),
    ("工单库落在卷内", "TICKETS_DB_PATH: /app/data/tickets.db" in compose),
    (".dockerignore 用 **/*.db 排除嵌套库", "**/*.db" in di_patterns),
    (".dockerignore 未用裸 *.db（不跨目录）", "*.db" not in di_patterns),
]
print("=" * 66)
print("  部署一致性自检")
print("=" * 66)
for name, ok in checks:
    print(f"    [{'OK' if ok else 'NO'}] {name}")
print()
print("  全部通过：" , all(ok for _, ok in checks))

  部署一致性自检
    [OK] core 镜像拷了 eval 包
    [OK] core.main 顶层依赖 eval
    [OK] core 镜像不拷 tests
    [OK] frontend 用 npm ci
    [OK] frontend 源码路径带前缀
    [OK] compose 引用两个 Dockerfile
    [OK] compose 挂 data 卷
    [OK] 工单库落在卷内
    [OK] .dockerignore 用 **/*.db 排除嵌套库
    [OK] .dockerignore 未用裸 *.db（不跨目录）

  全部通过： True


### 10.3 依赖与忽略清单

- `requirements.txt`：给关键依赖加**兼容上限**。原因是 `crew_agents.py`
  依赖 CrewAI 的**内部路径**（`crewai.llms.providers.openai.*`、`crewai.flow`），
  跨大版本极易破；本地引擎（PaddleOCR/funasr）列为**可选依赖**，未装时切 `api` 供应商。
- `.dockerignore`：排除 `.env`、`.venv`、`node_modules`、`data`、`docs`，
  以及 **`**/*.db`**（必须跨目录匹配，才能排除 `mcp_servers/.../tickets.db`）。

In [27]:
# ============================================================
# Cell 27：requirements.txt 与 .dockerignore 走读
# ============================================================
print("=" * 66); print("  requirements.txt"); print("=" * 66)
print(Path("requirements.txt").read_text(encoding="utf-8"))
print("=" * 66); print("  .dockerignore"); print("=" * 66)
print(Path(".dockerignore").read_text(encoding="utf-8"))

  requirements.txt
# ============ Web 框架 ============
fastapi>=0.115.0
uvicorn[standard]>=0.30.0
python-multipart>=0.0.9
websockets>=12.0

# ============ 配置与校验 ============
pydantic>=2.7.0
pydantic-settings>=2.3.0

# ============ 模型调用（OpenAI 兼容协议）============
openai>=1.40.0
httpx>=0.27.0

# ============ Agent 编排 ============
# 上限必要：crew_agents.py 依赖 provider 内部路径（crewai.llms.providers.openai.*）
# 与 crewai.flow / crewai.tools，跨大版本易破；下限对齐本机实测版本
crewai>=1.15,<2

# ============ MCP 协议 ============
mcp>=1.28,<2
fastmcp>=3.4,<4

# ============ 多模态 RAG ============
chromadb>=1.1,<2
numpy>=1.26.0
pymupdf>=1.24.0          # PDF 文本/图片抽取与整页渲染
python-docx>=1.1.0       # docx 解析
Pillow>=10.3.0           # 图片降采样

# ============ 多模态感知 ============
# 上限必要：代码依赖 7.x 的 WordBoundary 时间戳（逐词字幕）
edge-tts>=7.0,<8         # TTS（本地、免费、零部署）
# 说明：本地 OCR / ASR 引擎（paddleocr + paddlepaddle / funasr）为**可选依赖**，未随本清单安装；
#       未安装时请将 OCR_PROVIDER / ASR_PROVIDER 设为 api（复用 VLM / 在线语音识别），见 .env.example

# ============ 定时

---
## Part 11：实战任务 + 随堂练习（含答案）
---

### 11.1 实战任务

| 任务 | 做法 | 观察点 |
|------|------|--------|
| **跑评测** | 管理端「评测」页勾选四套件，点开始 | 先返回 run_id，进度随轮询推进；结束后看 PASS/FAIL/SKIP |
| **定位未达标** | 从报告里找一项 FAIL | 看 `metrics` 与逐条 `items`，给出优化方向 |
| **设基线** | 把一次 done 的评测「设为基线」 | 后续运行详情能看到 Δ 对比 |
| **一键起全栈** | 仓库根执行 `docker compose -f deploy/docker-compose.yml up -d` | :8000 与 :3000 均可访问 |
| **验证持久化** | 建会话/上传知识库 → 重启容器 | ChromaDB / 媒体 / 工单库数据仍在 |
| **回归** | `pytest -q` | 全绿即封版门槛 |

### 11.2 常见坑与答疑

| 现象 | 根因 | 纠法 |
|------|------|------|
| `.env` 里值后带 `# 注释`，启动报 `ValidationError` | `pydantic-settings` **不剥行内注释** | 注释另起一行；本次已用**绝对路径**加载 `.env` 避免 CWD 漂移 |
| 容器启动即 `No module named 'eval'` | `core.main` 顶层导入 `eval`，但镜像漏拷 | `Dockerfile.core` 必须 `COPY eval ./eval`（已固化成测试） |
| 重启容器后数据全空 | 持久化路径没落卷内 | `DB_PATH`/`CHROMA_PATH`/`MEDIA_DIR`/`TICKETS_DB_PATH` 全指到 `/app/data` |
| 评测记录永远"运行中"删不掉 | 进程被杀，`running` 没收敛 | 启动时 `recover_stale_runs()` 收敛为 failed |
| 环境没准备好却报"质量不达标" | 把 SKIP 当 FAIL | `verdict(None, ...)` 返回 SKIP；检索缺金标来源走 `skipped(...)` |
| 改了 `crewai` 版本后 Agent 直接崩 | 依赖了 CrewAI 内部路径 | `requirements.txt` 加兼容上限（`crewai>=1.15,<2`） |

### 11.3 随堂练习一：为什么"没测到"不能判 FAIL？

假设知识库还没加载演示数据，检索套件一条都命中不了。如果直接判 `FAIL`，
会带来什么问题？正确做法是什么？

In [28]:
# ============================================================
# Cell 28：练习一参考思路（含答案）
# ============================================================
answer = """
问题：如果知识库没加载就把检索套件判 FAIL，会把【环境问题】误报成【质量问题】。
后果：
  1. 团队以为'检索能力不行'，去调模型/改分块，白费功夫；
  2. 连续几次 FAIL 后，大家开始不信任报告（'反正都是红的'），评测失去意义；
  3. 无法区分'这次真的退步了'还是'这次环境没准备好'，回归对比失效。

正确做法（本项目已在代码里落地）：
  1. verdict(value=None, ...) -> "SKIP"：指标没测到就不判 FAIL；
  2. 检索套件先做【前置检查】——金标来源是否都在已索引文档里，
     缺了就 skipped(...) 返回 SKIP，并把缺失文件写进 notes 提示'请先加载演示知识库'；
  3. 报告页把 SKIP 与 FAIL 分开呈现，SKIP 附'请先完成前置操作'的指引。

一句话：SKIP = 没法测（环境/依赖问题）；FAIL = 确实测了但不达标（质量问题）。
"""
print(answer)


问题：如果知识库没加载就把检索套件判 FAIL，会把【环境问题】误报成【质量问题】。
后果：
  1. 团队以为'检索能力不行'，去调模型/改分块，白费功夫；
  2. 连续几次 FAIL 后，大家开始不信任报告（'反正都是红的'），评测失去意义；
  3. 无法区分'这次真的退步了'还是'这次环境没准备好'，回归对比失效。

正确做法（本项目已在代码里落地）：
  1. verdict(value=None, ...) -> "SKIP"：指标没测到就不判 FAIL；
  2. 检索套件先做【前置检查】——金标来源是否都在已索引文档里，
     缺了就 skipped(...) 返回 SKIP，并把缺失文件写进 notes 提示'请先加载演示知识库'；
  3. 报告页把 SKIP 与 FAIL 分开呈现，SKIP 附'请先完成前置操作'的指引。

一句话：SKIP = 没法测（环境/依赖问题）；FAIL = 确实测了但不达标（质量问题）。



### 11.4 随堂练习二：新增一个评测套件要改哪些地方？

产品说："想加一个'意图识别准确率'的评测套件。"

请说出：要新建什么、要登记到哪里、runner 要不要改？

In [29]:
# ============================================================
# Cell 29：练习二参考思路（含答案）
# ============================================================
answer = """
按'插件式'设计，只需三步，runner 一行都不用改：

1. 新建 eval/suites/intent.py，暴露三个约定接口：
     SUITE = "intent"
     TITLE = "意图识别准确率"
     async def run(limit=None) -> dict:
         cases = load_cases("intent_cases.jsonl", limit)      # 复用公共工具
         items = await map_limited(cases, _one)               # 复用限流并发
         metrics_map = {"accuracy": ...}
         return build_result(SUITE, TITLE, metrics_map, items,
                             primary="accuracy", primary_threshold=..., lower_is_better=False)

2. 在 eval/suites/__init__.py 登记：
     from eval.suites import asr, intent, retrieval, tts, vlm
     SUITES = {..., intent.SUITE: intent}
     DEFAULT_ORDER = (asr.SUITE, tts.SUITE, vlm.SUITE, retrieval.SUITE, intent.SUITE)

3. 新建数据集 eval/datasets/intent_cases.jsonl（一行一个用例）。

runner 不用改：它只遍历 run["suites"] 并从 SUITES 取模块；
管理端 /eval/suites 也会自动把新套件列出来。
——这就是'runner 只认约定接口、不关心内部实现'带来的可扩展性。
"""
print(answer)


按'插件式'设计，只需三步，runner 一行都不用改：

1. 新建 eval/suites/intent.py，暴露三个约定接口：
     SUITE = "intent"
     TITLE = "意图识别准确率"
     async def run(limit=None) -> dict:
         cases = load_cases("intent_cases.jsonl", limit)      # 复用公共工具
         items = await map_limited(cases, _one)               # 复用限流并发
         metrics_map = {"accuracy": ...}
         return build_result(SUITE, TITLE, metrics_map, items,
                             primary="accuracy", primary_threshold=..., lower_is_better=False)

2. 在 eval/suites/__init__.py 登记：
     from eval.suites import asr, intent, retrieval, tts, vlm
     SUITES = {..., intent.SUITE: intent}
     DEFAULT_ORDER = (asr.SUITE, tts.SUITE, vlm.SUITE, retrieval.SUITE, intent.SUITE)

3. 新建数据集 eval/datasets/intent_cases.jsonl（一行一个用例）。

runner 不用改：它只遍历 run["suites"] 并从 SUITES 取模块；
管理端 /eval/suites 也会自动把新套件列出来。
——这就是'runner 只认约定接口、不关心内部实现'带来的可扩展性。



### 11.5 随堂练习三：为什么 ChromaDB 嵌入式不能开多 worker？

请结合部署配置回答：如果 `core` 服务用 `uvicorn --workers 4` 启动，会发生什么？
`docker-compose.yml` 里是怎么规避的？

In [30]:
# ============================================================
# Cell 30：练习三参考思路（含答案）
# ============================================================
answer = """
原因：ChromaDB 嵌入式（PersistentClient）是【单进程独占】的本地数据库——
它直接以文件形式持有 data/chroma 目录。若启动多个 worker：
  1. 多个进程同时打开同一份 Chroma 数据，容易锁冲突/数据损坏；
  2. 每个 worker 各持一份内存连接，写入互相不可见，检索结果不一致；
  3. 进程内单例（ws_manager / 连接表 / 评测任务注册表 / MCP 桥）也会分裂——
     用户连到 worker A、推送却可能发生在 worker B，消息就会丢。

本项目怎么规避：
  - Dockerfile.core 的 CMD 用 uvicorn 且【不带 --workers】，即单 worker；
  - docker-compose.yml 里 core 不设多副本，restart: unless-stopped 但仍是单实例；
  - 代码注释与文档都明确写了'core 必须单实例运行'。

如果要水平扩展：需把向量库换成客户端-服务端形态（如 Chroma/Zdrant 服务端），
把连接表与任务注册表迁移到 Redis 等共享存储——这是架构级改动，不是加个参数。
"""
print(answer)


原因：ChromaDB 嵌入式（PersistentClient）是【单进程独占】的本地数据库——
它直接以文件形式持有 data/chroma 目录。若启动多个 worker：
  1. 多个进程同时打开同一份 Chroma 数据，容易锁冲突/数据损坏；
  2. 每个 worker 各持一份内存连接，写入互相不可见，检索结果不一致；
  3. 进程内单例（ws_manager / 连接表 / 评测任务注册表 / MCP 桥）也会分裂——
     用户连到 worker A、推送却可能发生在 worker B，消息就会丢。

本项目怎么规避：
  - Dockerfile.core 的 CMD 用 uvicorn 且【不带 --workers】，即单 worker；
  - docker-compose.yml 里 core 不设多副本，restart: unless-stopped 但仍是单实例；
  - 代码注释与文档都明确写了'core 必须单实例运行'。

如果要水平扩展：需把向量库换成客户端-服务端形态（如 Chroma/Zdrant 服务端），
把连接表与任务注册表迁移到 Redis 等共享存储——这是架构级改动，不是加个参数。



### 11.6 本节小结

| 环节 | 文件 | 核心设计 |
|------|------|----------|
| 指标库 | `eval/metrics.py` | 纯函数、可离线单测；中文数字/全角/小数点归一化；CER/检索/字段/描述四类 |
| 公共流水线 | `eval/suites/_common.py` | jsonl 逐行容错、信号量限流、`verdict` 区分 SKIP/FAIL、统一装配 |
| 四套件 | `eval/suites/{asr,tts,vlm,retrieval}.py` | 插件式（SUITE/TITLE/run）；ASR 用 TTS 自建主集；VLM 两类出口联合判定；检索双模式对照 |
| 数据与素材 | `eval/datasets/*.jsonl`、`eval/images.py` | jsonl 容错、合成图确定性生成（无隐私） |
| 编排 | `eval/runner.py` | 先返回 run_id、后台执行、单套件失败降级、取消显式收尾 |
| 持久化 | `eval/store.py` | 文件式、原子写、run_id 白名单、`recover_stale_runs` |
| 评判 | `eval/judges.py` | LLM-as-judge 幻觉检测；解析失败按"不可用"跳过，不臆断 |
| 接口 | `core/api/admin_api.py` | `/api/admin/eval/*`；running 不可删；仅 done 可设基线 |
| 回归测试 | `tests/**` | 纯函数离线可测；外部依赖用假对象；部署一致性也固化为测试 |
| 部署 | `deploy/**` | 两镜像 + Compose；构建上下文=仓库根；data 卷持久化；core 单 worker |
| 依赖 | `requirements.txt`、`.dockerignore` | 关键依赖加兼容上限；`**/*.db` 跨目录排除 |

### 11.7 交付物 / 验收

- **评测报告**：四套件（ASR/TTS/VLM/检索）可运行、判定明确、可设基线对比；
- **回归测试**：`pytest -q` 全绿（交付门槛）；
- **容器化部署**：`docker compose -f deploy/docker-compose.yml up -d` 起全栈，
  :8000 / :3000 可访问；
- **数据持久化**：重启容器后 ChromaDB、媒体、业务库、工单库数据保留；
- **交付文档**：产品说明 / 技术架构 / 部署运维手册 / 评测报告 / 演示脚本。

### 11.8 六节课一路走来

```
第34节  架构与配置       —— 立骨架：分层、能力四元组、启动自检
第35节  多模态感知       —— 会听会说会看图：ASR/TTS/OCR/VLM + 媒体鉴权
第36节  多模态 RAG       —— 知识从哪来：解析→分块→文本优先嵌入→检索→装配
第37节  多智能体协同     —— 会分工会转人：CrewAI 三角色 + 转人工 + MCP 工单
第38节  三端交互系统     —— 能上手用：Vue 三端 + WebSocket 实时链路与稳定性
第39节  评测部署交付     —— 可交付：量化评测 + 回归测试 + 容器化 + 文档
```

---

## 📚 参考资料

- 代码：`eval/metrics.py` —— 评测指标库（纯函数）
- 代码：`eval/suites/_common.py` —— 公共流水线
- 代码：`eval/suites/{asr,tts,vlm,retrieval}.py` —— 四个评测套件
- 代码：`eval/{runner,store,judges,images}.py` —— 编排 / 持久化 / 评判 / 素材
- 代码：`eval/suites/__init__.py` —— 套件注册表（插件式）
- 代码：`core/api/admin_api.py` —— `/api/admin/eval/*` 接口
- 代码：`deploy/Dockerfile.core` / `Dockerfile.frontend` / `docker-compose.yml`
- 文件：`requirements.txt`、`.dockerignore`
- 测试：`tests/`（7 个文件）

> 🎉 恭喜你完成了本项目的全部课程！
> 到这里，你不仅知道代码怎么写，更知道
> **怎么证明它好用（可量化）、怎么保证它不退化（可回归）、怎么把它交出去（可复现）**。